# 1. Setup

## 1.1: Install packages

## 1.1.1: Configuration

**All training and dataset parameters are centralized in `config.json`.**

This single source of truth is used by both this notebook and `training_args.py` (which exports parameters to LaTeX for the paper). To modify any training parameters, update `config.json` instead of hardcoding values here.


In [1]:
# !pip install --upgrade chess datasets huggingface_hub scikit-learn zstandard tokenizers transformers torch torchvision torchaudio 


In [1]:
import torch
DEVICE = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
DEVICE

device(type='cpu')

In [ ]:
from huggingface_hub import notebook_login, login
from dotenv import load_dotenv
from os import environ

load_dotenv()
login(token=environ["HF_TOKEN"])

## 1.2: Dataset retrieval

Dataset source: https://huggingface.co/datasets/austindavis/lichess-uci

In [ ]:
from datasets import load_dataset, Dataset, DatasetDict
import json

# Load configuration from centralized config.json
with open("config.json", "r") as file:
    config = json.load(file)
    print(config)

ds = load_dataset("austindavis/lichess-uci", "201505")["train"]

# First cut: split off test set — this is quarantined until the end
train_val = ds.train_test_split(test_size=0.05, seed=42)

# Second cut: split train into train and validation
train_val_split = train_val["train"].train_test_split(test_size=0.05, seed=42)

final_splits = {
    "train": train_val_split["train"],
    "val":   train_val_split["test"],
    "test":  train_val["test"],
}

ds = DatasetDict(final_splits)
# ds


README.md: 0.00B [00:00, ?B/s]

data/201505/train-00000-of-00003.parquet:   0%|          | 0.00/174M [00:00<?, ?B/s]

data/201505/train-00001-of-00003.parquet:   0%|          | 0.00/174M [00:00<?, ?B/s]

data/201505/train-00002-of-00003.parquet:   0%|          | 0.00/175M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2137556 [00:00<?, ? examples/s]

In [5]:
ds["train"][0]

{'Event': 'Rated Bullet tournament https://lichess.org/tournament/MJHHFtgM',
 'Site': 'PPEvEQ91',
 'White': 'Suninheart',
 'Black': 'aila',
 'Result': '0-1',
 'UTCDate': datetime.date(2015, 5, 14),
 'UTCTime': datetime.time(21, 35, 10),
 'WhiteElo': 0,
 'BlackElo': 0,
 'WhiteRatingDiff': -9.0,
 'BlackRatingDiff': 4.0,
 'ECO': 'D05',
 'Opening': "Queen's Pawn Game, Zukertort Variation",
 'TimeControl': '120+0',
 'Termination': 'Normal',
 'Transcript': 'd2d4 d7d5 e2e3 e7e6 g1f3 g8f6 b1d2 c7c5 b2b3 c5d4 e3d4 b8c6 c1b2 f8e7 a2a3 e8g8 f1d3 a7a6 e1g1 c8d7 f3e5 a8b8 d2f3 c6e5 d4e5 f6e8 c2c4 d5c4 b3c4 f7f6 e5f6 e7f6 d1c2 h7h6 d3h7 g8h8 b2f6 d8f6 h7e4 e8d6 a1d1 d6e4 c2e4 d7c6 e4g4 c6f3 g2f3 f6f3 g4f3 f8f3 g1g2 f3a3 d1d2 a3c3 f1d1 c3c4 d2d8 b8d8 d1d8 h8h7 d8b8 b7b5 b8a8 c4a4 g2f3 b5b4 a8b8 a6a5 f3e3 a4a2 f2f4 a2h2 e3e4 h2h3 e4e5 b4b3 e5e6 a5a4 e6f5 a4a3 f5g4 h3c3 g4f5 a3a2 b8a8 b3b2 a8a2 b2b1Q f5e5 b1a2 f4f5 a2e2 e5f4'}

In [6]:
RESULT_STRINGS = {"1-0", "0-1", "1/2-1/2", "*"}

def clean_transcript(sample):
    cleaned_transcripts = []
    cleaned_moves = []
    num_moves = []
    
    for transcript in sample["Transcript"]:
        # Remove result string from end if present
        moves = transcript.lower().split()
        if moves and moves[-1] in RESULT_STRINGS:
            moves = moves[:-1]
        cleaned_transcripts.append(" ".join(moves))
        cleaned_moves.append(moves)
        num_moves.append(len(moves))
    
    return {
        "Transcript": cleaned_transcripts,
        "Moves": cleaned_moves,
        "Number of Moves": num_moves
    }

ds = ds.map(clean_transcript, batched=True, num_proc=4, desc="Cleaning transcripts")
ds["train"][0]

Cleaning transcripts (num_proc=4):   0%|          | 0/1929144 [00:00<?, ? examples/s]

Cleaning transcripts (num_proc=4):   0%|          | 0/101534 [00:00<?, ? examples/s]

Cleaning transcripts (num_proc=4):   0%|          | 0/106878 [00:00<?, ? examples/s]

{'Event': 'Rated Bullet tournament https://lichess.org/tournament/MJHHFtgM',
 'Site': 'PPEvEQ91',
 'White': 'Suninheart',
 'Black': 'aila',
 'Result': '0-1',
 'UTCDate': datetime.date(2015, 5, 14),
 'UTCTime': datetime.time(21, 35, 10),
 'WhiteElo': 0,
 'BlackElo': 0,
 'WhiteRatingDiff': -9.0,
 'BlackRatingDiff': 4.0,
 'ECO': 'D05',
 'Opening': "Queen's Pawn Game, Zukertort Variation",
 'TimeControl': '120+0',
 'Termination': 'Normal',
 'Transcript': 'd2d4 d7d5 e2e3 e7e6 g1f3 g8f6 b1d2 c7c5 b2b3 c5d4 e3d4 b8c6 c1b2 f8e7 a2a3 e8g8 f1d3 a7a6 e1g1 c8d7 f3e5 a8b8 d2f3 c6e5 d4e5 f6e8 c2c4 d5c4 b3c4 f7f6 e5f6 e7f6 d1c2 h7h6 d3h7 g8h8 b2f6 d8f6 h7e4 e8d6 a1d1 d6e4 c2e4 d7c6 e4g4 c6f3 g2f3 f6f3 g4f3 f8f3 g1g2 f3a3 d1d2 a3c3 f1d1 c3c4 d2d8 b8d8 d1d8 h8h7 d8b8 b7b5 b8a8 c4a4 g2f3 b5b4 a8b8 a6a5 f3e3 a4a2 f2f4 a2h2 e3e4 h2h3 e4e5 b4b3 e5e6 a5a4 e6f5 a4a3 f5g4 h3c3 g4f5 a3a2 b8a8 b3b2 a8a2 b2b1q f5e5 b1a2 f4f5 a2e2 e5f4',
 'Moves': ['d2d4',
  'd7d5',
  'e2e3',
  'e7e6',
  'g1f3',
  'g8f6',
  'b1d2

# 2. Tokenization

## 2.1 Vocabularization

There are a small number of legal moves in UCI notation. We construct a small vocabulary:

Now, we build actual tokens

In [7]:
from collections import Counter

# Special tokens first, then all moves sorted by frequency
# Sorting by frequency is a minor nicety — most common tokens
# get low IDs, which can matter for some embedding implementations
SPECIAL_TOKENS = [
    "[PAD]",        # 0 - padding (you may not need this with flat binary approach)
    "[START]",      # 1 - beginning of game
    "[CHECKMATE]",  # 2
    "[STALEMATE]",  # 3
    "[DRAW_REP]",   # 4 - threefold/fivefold repetition
    "[DRAW_50]",    # 5 - 50/75 move rule
    "[DRAW_MAT]",   # 6 - insufficient material
    "[RESIGN]",     # 7 - resignation
    "[TIMEOUT]",    # 8 - TIMEOUT
    "[UNK]",        # 9 - unknown/malformed move (safety net)
]

move_counter = Counter()
for moves in ds["train"]["Moves"]:
    move_counter.update(moves)

print(f"Unique moves after cleaning: {len(move_counter)}")
# Should be ~1966, and 1-0/0-1/1/2-1/2 should be gone
print(f"1-0 still in counter: {'1-0' in move_counter}")


# Build vocab: special tokens + all moves by descending frequency
all_moves_sorted = [move for move, _ in move_counter.most_common()]
vocab = SPECIAL_TOKENS + all_moves_sorted

token2id = {token: idx for idx, token in enumerate(vocab)}
id2token = {idx: token for token, idx in token2id.items()}

len(vocab)

Unique moves after cleaning: 1968
1-0 still in counter: False


1978

Build a token-to-id mapper, and inverse. We also save the vocab for convenience.

Token test

In [8]:
# Round-trip test — should print True
test_token = "e2e4"
assert id2token[token2id[test_token]] == test_token
print(f"vocab size: {len(vocab)}")
print(f"e2e4 is token {token2id['e2e4']}")
print(f"token 1 is {id2token[1]}")  # should be [START]
print(f"rarest move: {vocab[-1]} at index {len(vocab)-1}")

vocab size: 1978
e2e4 is token 12
token 1 is [START]
rarest move: d2c1b at index 1977


We count the types of game termination, and their results

In [9]:
from collections import Counter

term_result_counter = Counter(
    zip(ds["train"]["Termination"], ds["train"]["Result"])
)

for (term, result), count in term_result_counter.most_common():
    print(f"({term!r}, {result!r}): {count:,}")

('Normal', '1-0'): 639,837
('Normal', '0-1'): 579,087
('Time forfeit', '1-0'): 316,064
('Time forfeit', '0-1'): 305,911
('Normal', '1/2-1/2'): 53,901
('Time forfeit', '1/2-1/2'): 14,579
('Abandoned', '0-1'): 11,455
('Abandoned', '1-0'): 8,212
('Rules infraction', '1-0'): 96
('Abandoned', '1/2-1/2'): 2


In [10]:
# mass tokenize games taking advantage of ds.map() again

from typing import Any

TERM_MAP = {
    ("Normal",           "1-0"):     "[RESIGN]",
    ("Normal",           "0-1"):     "[RESIGN]",
    ("Normal",           "1/2-1/2"): "[DRAW_REP]",
    ("Time forfeit",     "1-0"):     "[TIMEOUT]",
    ("Time forfeit",     "0-1"):     "[TIMEOUT]",
    ("Time forfeit",     "1/2-1/2"): "[TIMEOUT]",
    ("Abandoned",        "1-0"):     "[RESIGN]",
    ("Abandoned",        "0-1"):     "[RESIGN]",
    ("Abandoned",        "1/2-1/2"): "[DRAW_REP]",
    ("Rules infraction", "1-0"):     "[RESIGN]",
}

import numpy as np

def make_tokenize_fn(token2id, TERM_MAP):
    START_ID = token2id["[START]"]
    UNK_ID   = token2id["[UNK]"]
    
    def tokenize_batch(batch):
        all_encoded = []
        
        for moves, termination, result in zip(
            batch["Moves"],
            batch["Termination"],
            batch["Result"]
        ):
            end_token = TERM_MAP.get((termination, result))
            
            if not moves or result == "*" or end_token is None:
                all_encoded.append(None)
                continue
            
            ids = [START_ID]
            for move in moves:
                ids.append(token2id.get(move, UNK_ID))
            ids.append(token2id[end_token])
            
            all_encoded.append(ids)
        
        return {"input_ids": all_encoded}
    
    return tokenize_batch

tokenize_fn = make_tokenize_fn(token2id, TERM_MAP)

ds = ds.map(
    tokenize_fn,
    batched=True,
    # batch_size=10_000,
    num_proc=4,       # Kaggle gives you multiple cores
    desc="Tokenizing"
)

Tokenizing (num_proc=4):   0%|          | 0/1929144 [00:00<?, ? examples/s]

Tokenizing (num_proc=4):   0%|          | 0/101534 [00:00<?, ? examples/s]

Tokenizing (num_proc=4):   0%|          | 0/106878 [00:00<?, ? examples/s]

## Tokenizer

We now build the actual tokenizer.

We use the WordLevel Tokenizer since every of our moves (like `e2e4`) need to be recorded in full, as well as the fact that moves are separated by spaces.

In [11]:
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace

tokenizer = Tokenizer(WordLevel(vocab=token2id, unk_token="[UNK]"))

tokenizer.pre_tokenizer = Whitespace()

tokenizer.add_special_tokens(SPECIAL_TOKENS)

tokenizer.save("/kaggle/working/tokenizer.json")
print("Tokenizer saved.")

Tokenizer saved.


Tokenizer 

In [12]:
enc = tokenizer.encode("e2e4 e7e5 g1f3 b8c6")
print(enc.tokens)   # ['e2e4', 'e7e5', 'g1f3', 'b8c6']
print(enc.ids)      # [some integers matching your token2id]

# Round trip
decoded = tokenizer.decode(enc.ids)
print(decoded)      # 'e2e4 e7e5 g1f3 b8c6'

# Reload test

# assert tok2.encode("e2e4").ids == tokenizer.encode("e2e4").ids
print("Reload OK")

['e2e4', 'e7e5', 'g1f3', 'b8c6']
[12, 21, 10, 17]
e2e4 e7e5 g1f3 b8c6
Reload OK


In [13]:
# Encode a game prefix to feed to the model
moves_so_far = "e2e4 e7e5 g1f3"
ids = tokenizer.encode(moves_so_far).ids
# prepend [START] manually since encode() doesn't add it automatically
ids = [token2id["[START]"]] + ids
print(ids)

# Decode model output back to moves
predicted_id = 423  # whatever the model sampled
predicted_move = tokenizer.id_to_token(predicted_id)  # "b8c6"
print(predicted_move)

[1, 12, 21, 10]
b3b2


In [14]:
print("Special Tokens:", [token for token in token2id if token.startswith("[")])

Special Tokens: ['[PAD]', '[START]', '[CHECKMATE]', '[STALEMATE]', '[DRAW_REP]', '[DRAW_50]', '[DRAW_MAT]', '[RESIGN]', '[TIMEOUT]', '[UNK]']


In [15]:
from transformers import PreTrainedTokenizerFast

hf_tokenizer = PreTrainedTokenizerFast(
    tokenizer_file = "/kaggle/working/tokenizer.json",
    bos_token      = "[START]",
    eos_token      = "[RESIGN]",
    pad_token      = "[PAD]",
    unk_token      = "[UNK]",
)

# Verify it round-trips correctly
test = hf_tokenizer.encode("e2e4 e7e5 g1f3")
print(test)
print(hf_tokenizer.decode(test))

from transformers import DataCollatorForLanguageModeling

hf_tokenizer.pad_token = hf_tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(hf_tokenizer, mlm=False)
data_collator

[12, 21, 10]
e2e4 e7e5 g1f3


DataCollatorForLanguageModeling(tokenizer=TokenizersBackend(name_or_path='', vocab_size=1978, model_max_length=1000000000000000019884624838656, padding_side='right', truncation_side='right', special_tokens={'bos_token': '[START]', 'eos_token': '[RESIGN]', 'unk_token': '[UNK]', 'pad_token': '[RESIGN]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("[START]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("[CHECKMATE]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("[STALEMATE]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	4: AddedToken("[DRAW_REP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	5: AddedToken("[DRAW_50]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	6: AddedToken("[DRAW_MAT]

## Model

In this paper, we use the GPT-2 model.

In [16]:
from transformers import GPT2Config, GPT2LMHeadModel

config = GPT2Config(
    vocab_size          = len(vocab),      # 1980
    n_positions         = 256,             # max sequence length
    n_embd              = 256,             # d_model
    n_layer             = 6,               # transformer blocks
    n_head              = 8,               # attention heads
    n_inner             = 1024,            # MLP hidden dim (4 * n_embd)
    activation_function = "gelu_new",
    resid_pdrop         = 0.1,
    embd_pdrop          = 0.1,
    attn_pdrop          = 0.1,
    bos_token_id        = token2id["[START]"],
    eos_token_id        = [
        token2id["[RESIGN]"],
        token2id["[TIMEOUT]"],
        token2id["[DRAW_REP]"],
        token2id["[CHECKMATE]"],
        token2id["[STALEMATE]"],
        token2id["[DRAW_MAT]"],
        token2id["[DRAW_50]"],
    ],
    attn_implementation = "sdpa",          # the one good part from that line
)

# Randomly initialized — no pretrained weights
model = GPT2LMHeadModel(config)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
# Should be ~8-10M

Parameters: 5,310,976


In [17]:
ds

DatasetDict({
    train: Dataset({
        features: ['Event', 'Site', 'White', 'Black', 'Result', 'UTCDate', 'UTCTime', 'WhiteElo', 'BlackElo', 'WhiteRatingDiff', 'BlackRatingDiff', 'ECO', 'Opening', 'TimeControl', 'Termination', 'Transcript', 'Moves', 'Number of Moves', 'input_ids'],
        num_rows: 1929144
    })
    val: Dataset({
        features: ['Event', 'Site', 'White', 'Black', 'Result', 'UTCDate', 'UTCTime', 'WhiteElo', 'BlackElo', 'WhiteRatingDiff', 'BlackRatingDiff', 'ECO', 'Opening', 'TimeControl', 'Termination', 'Transcript', 'Moves', 'Number of Moves', 'input_ids'],
        num_rows: 101534
    })
    test: Dataset({
        features: ['Event', 'Site', 'White', 'Black', 'Result', 'UTCDate', 'UTCTime', 'WhiteElo', 'BlackElo', 'WhiteRatingDiff', 'BlackRatingDiff', 'ECO', 'Opening', 'TimeControl', 'Termination', 'Transcript', 'Moves', 'Number of Moves', 'input_ids'],
        num_rows: 106878
    })
})

In [18]:
bad_rows = ds["train"].filter(lambda x: x["input_ids"] is None)

print(bad_rows)
print(bad_rows[0])

Filter:   0%|          | 0/1929144 [00:00<?, ? examples/s]

Dataset({
    features: ['Event', 'Site', 'White', 'Black', 'Result', 'UTCDate', 'UTCTime', 'WhiteElo', 'BlackElo', 'WhiteRatingDiff', 'BlackRatingDiff', 'ECO', 'Opening', 'TimeControl', 'Termination', 'Transcript', 'Moves', 'Number of Moves', 'input_ids'],
    num_rows: 17693
})
{'Event': 'Rated Blitz tournament https://lichess.org/tournament/gbgNEG7T', 'Site': 'JKVo5BnX', 'White': 'ow055', 'Black': 'Bu1let', 'Result': '0-1', 'UTCDate': datetime.date(2015, 5, 6), 'UTCTime': datetime.time(8, 12, 27), 'WhiteElo': 0, 'BlackElo': 0, 'WhiteRatingDiff': None, 'BlackRatingDiff': None, 'ECO': '?', 'Opening': '?', 'TimeControl': '300+0', 'Termination': 'Abandoned', 'Transcript': '', 'Moves': [], 'Number of Moves': 0, 'input_ids': None}


## Chunking

In [19]:
BLOCK_SIZE = 256

ds = ds.filter(lambda x: x["input_ids"] is not None)

def group_texts(batch):
    # Concatenate all sequences in the batch into one flat list
    concatenated = sum(batch["input_ids"], [])
    total_length = len(concatenated)
    
    # Drop the remainder
    total_length = (total_length // BLOCK_SIZE) * BLOCK_SIZE
    
    # Cut into chunks
    chunks = [
        concatenated[i : i + BLOCK_SIZE]
        for i in range(0, total_length, BLOCK_SIZE)
    ]
    
    return {"input_ids": chunks}

train_chunked = ds["train"].map(
    group_texts,
    batched=True,
    batch_size=1000,
    remove_columns=ds["train"].column_names,
    desc="Chunking train",
)

val_chunked = ds["val"].map(
    group_texts,
    batched=True,
    batch_size=1000,
    remove_columns=ds["val"].column_names,
    desc="Chunking val",
)

print(train_chunked)
print(val_chunked)

Filter:   0%|          | 0/1929144 [00:00<?, ? examples/s]

Filter:   0%|          | 0/101534 [00:00<?, ? examples/s]

Filter:   0%|          | 0/106878 [00:00<?, ? examples/s]

Chunking train:   0%|          | 0/1911451 [00:00<?, ? examples/s]

Chunking val:   0%|          | 0/100608 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids'],
    num_rows: 509381
})
Dataset({
    features: ['input_ids'],
    num_rows: 26816
})


## Training? 

In [ ]:
# 1. Chunk the datasets (if not done yet)

from dataclasses import dataclass
from transformers import TrainingArguments, Trainer
from typing import List, Dict, Any
import torch
import json

# Load config
with open("config.json", "r") as f:
    config = json.load(f)

# 2. Collator
@dataclass
class FixedLengthCollator:
    def __call__(self, examples: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        input_ids = torch.tensor(
            [ex["input_ids"] for ex in examples],
            dtype=torch.long
        )
        return {
            "input_ids": input_ids,
            "labels":    input_ids.clone(),
        }

collator = FixedLengthCollator()
# collator = FixedLengthCollator()

# 3. Trainer - load training arguments from config.json

training_args = TrainingArguments(
    output_dir                  = "/kaggle/working/chess_gpt",
    num_train_epochs            = config["training"]["num_epochs"],
    per_device_train_batch_size = config["training"]["per_device_train_batch_size"],
    per_device_eval_batch_size  = config["training"]["per_device_eval_batch_size"],
    fp16                        = config["training"]["fp16"],
    learning_rate               = config["training"]["learning_rate"],
    weight_decay                = config["training"]["weight_decay"],
    adam_beta1                  = config["training"]["adam_beta1"],
    adam_beta2                  = config["training"]["adam_beta2"],
    max_grad_norm               = config["training"]["max_grad_norm"],
    lr_scheduler_type           = config["training"]["lr_scheduler_type"],
    warmup_steps                = config["training"]["warmup_steps"],
    eval_strategy               = "steps",
    eval_steps                  = 500,
    save_strategy               = "steps",
    save_steps                  = 500,
    save_total_limit            = 3,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    logging_steps               = 50,
    report_to                   = "none",
    dataloader_num_workers      = 2,
    seed                        = 42,
)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = train_chunked,
    eval_dataset  = val_chunked,
    data_collator = collator,
    processing_class = hf_tokenizer
)

trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 7, 'pad_token_id': 7}.
We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
You may ignore this warning if your `pad_token_id` (7) is identical to the `bos_token_id` (1), `eos_token_id` (7), or the `sep_token_id` (None), and your input is not padded.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
500,4.958247,4.811193
1000,4.190368,4.000440
1500,3.793643,3.587984
2000,3.555676,3.349410
2500,3.406116,3.198514
3000,3.289634,3.086565


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Yes, make sure we don't forget it this time.

In [ ]:
trainer.push_to_hub()

In [ ]:
raw_model = trainer.model
raw_model.eval()

start_token = torch.tensor(
    [[token2id["[START]"]]],
    dtype=torch.long
).to(DEVICE)

with torch.no_grad():
    output = raw_model.generate(
        start_token,
        max_new_tokens = 150,
        do_sample      = True,
        temperature    = 0.5,
        pad_token_id   = token2id["[PAD]"],
    )

decoded = [id2token.get(t, f"<{t}>") for t in output[0].tolist()]
print(" ".join(decoded))

Evaluate the generated game!

In [ ]:
import chess
import chess.pgn

game = chess.pgn.Game()
board = chess.Board()
for ply in decoded[1:-1]:
    move = chess.Move.from_uci(ply)
    assert board.is_legal(move), f"Error: can't make move {ply} in position {board.fen()}"
    board.push(move)
    print(move)
    